# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR² dataset package](https://doi.org/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library. All entities (record sets, fields, columns) are referenced by their unique `@id` based on the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata (this also initializes record set index)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list all record sets found in the metadata, and for each, show their fields (columns) and IDs.


In [ ]:
print("Record Sets:`@id` | Name | Description | Example field `@id`s")
overview = []
for rs in metadata.record_sets:
    print(f"- {rs.id} | {rs.name} | {rs.description if rs.description else '-'}")
    field_ids = [field.id for field in rs.fields]
    if len(field_ids) > 0:
        print(f"    Fields: {field_ids[:4]}{' ...' if len(field_ids) > 4 else ''}")
    overview.append({
        'record_set_id': rs.id,
        'record_set_name': rs.name,
        'fields': field_ids
    })
if not overview:
    print("No record sets were found in the Croissant metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

> **Note:** Replace `record_set_id` with the desired record set `@id` as listed above. For demonstration, we will extract all record sets found in the metadata.

In [ ]:
# Collect all available record set `@id`s from the metadata
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}
_errors = []
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded Record Set: {record_set_id} with {len(df)} records and columns: {df.columns.tolist()}")
        else:
            print(f"Record Set {record_set_id} exists but contains no records.")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")
        _errors.append((record_set_id, str(e)))

# As a demonstration, pick the first loaded DataFrame for further analysis
main_record_set_id = list(dataframes.keys())[0] if dataframes else None
if main_record_set_id:
    print(f"\nExample fields in DataFrame from {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- We demonstrate with a numeric field and a group field, both referenced by their `@id`s. Adjust these variables according to your dataset overview or replace them with the respective IDs from your data if desired.

In [ ]:
# For demonstration, select the first numeric column from the main record set
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Try to find a numeric column programmatically
    numeric_field_candidates = df.select_dtypes(include=[np.number]).columns
    if len(numeric_field_candidates) > 0:
        numeric_field = numeric_field_candidates[0]
        print(f"First numeric field in this record set (@id): {numeric_field}")
    else:
        print("No numeric fields detected. Please select a field manually.")
        numeric_field = None

    # Filtering
    threshold = 10
    if numeric_field and numeric_field in df:
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where {numeric_field} > {threshold}: {len(filtered_df)} records")

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            filtered_df[numeric_field].std()
        )
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a non-numeric group field
        group_field_candidates = df.select_dtypes(include=[object, "category"]).columns
        group_field = [col for col in group_field_candidates if col != numeric_field]
        group_field = group_field[0] if group_field else None
        if group_field and group_field in filtered_df:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped mean by {group_field}:")
            display(grouped_df.head())
        else:
            print("No grouping field available or detected. Skipping group by.")
    else:
        print(f"No numeric field suitable for filtering in {main_record_set_id}.")
else:
    print("No record set loaded for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we show how to plot the distribution of a numeric field (if present) and the relationship to a group (if available).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if main_record_set_id and numeric_field and numeric_field in df:
    # Distribution plot
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='tab:blue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Box plot by group (if group field present)
    if group_field and group_field in df:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df, showmeans=True)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

- This notebook illustrated how to use the `mlcroissant` library to load metadata, extract records, examine schema entities by their `@id`, and explore/visualize tabular data from the Croissant-based FAIR² dataset.
- For a full analysis, consider reviewing all record sets and all their fields as described in the metadata.
- The approach here can be reused for other Croissant-schema datasets by referencing field and record set IDs appropriately.